# 3-stage: слайд-шоу предсказаний модели

Положи модель в `3-stage/best_model.pth`. КТ-срезы и маски остаются в `preprocessed_npy/` или `prepared_npy/`.

Ноутбук загружает все срезы выбранного пациента и показывает интерактивное слайд-шоу. Каждый кадр состоит из трёх наложенных слоёв:

1. CT-срез — серый фон;
2. истинная маска — синяя область;
3. предсказанная маска — красная область.

Для текущего кадра показываются `Dice`, `Precision`, `Recall`.


## 1. Импорты и настройки

Поменяй `DATASET_DIR`, `SPLIT` и `CASE_ID`, если нужен другой пациент.


In [ ]:
from pathlib import Path
import importlib.util
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import torch
from IPython.display import display
import ipywidgets as widgets


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "2-stage" / "model.py").exists() and (candidate / "3-stage").exists():
            return candidate
    raise FileNotFoundError(
        "Не найден 2-stage/model.py. Запусти notebook из репозитория Lung-Tumor-Segmentation."
    )


REPO_ROOT = find_repo_root()
CHECKPOINT_PATH = REPO_ROOT / "3-stage" / "best_model.pth"

# Выбор серии срезов для слайд-шоу.
DATASET_DIR = REPO_ROOT / "preprocessed_npy"  # или REPO_ROOT / "prepared_npy"
SPLIT = "test"
CASE_ID = "lung_096"
INFERENCE_BATCH_SIZE = 8

IMAGE_DIR = DATASET_DIR / SPLIT / CASE_ID / "images"
MASK_DIR = DATASET_DIR / SPLIT / CASE_ID / "masks"

for path, label in [
    (CHECKPOINT_PATH, "model"),
    (IMAGE_DIR, "images directory"),
    (MASK_DIR, "masks directory"),
]:
    if not path.exists():
        raise FileNotFoundError(f"Не найден {label}: {path}")

MODEL_FILE = REPO_ROOT / "2-stage" / "model.py"
spec = importlib.util.spec_from_file_location("lung_unet_model", MODEL_FILE)
if spec is None or spec.loader is None:
    raise ImportError(f"Не удалось загрузить model.py: {MODEL_FILE}")
model_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_module)
build_model = model_module.build_model

image_paths = sorted(IMAGE_DIR.glob("*.npy"))
if not image_paths:
    raise FileNotFoundError(f"В папке нет .npy-срезов: {IMAGE_DIR}")

mask_paths = [MASK_DIR / path.name for path in image_paths]
missing_masks = [path for path in mask_paths if not path.exists()]
if missing_masks:
    raise FileNotFoundError(f"Не найдены маски для {len(missing_masks)} срезов. Первая: {missing_masks[0]}")

print("model:", CHECKPOINT_PATH)
print("case:", f"{SPLIT}/{CASE_ID}")
print("slices:", len(image_paths))


## 2. Вспомогательные функции


In [ ]:
def load_checkpoint(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def choose_device():
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def as_chw_float32(array: np.ndarray) -> np.ndarray:
    array = np.asarray(array)
    if array.ndim == 2:
        array = array[None, :, :]
    elif array.ndim == 3 and array.shape[0] == 1:
        pass
    elif array.ndim == 3 and array.shape[-1] == 1:
        array = np.moveaxis(array, -1, 0)
    else:
        raise ValueError(f"Ожидался массив [H, W] или [1, H, W], получено: {array.shape}")
    return array.astype(np.float32, copy=False)


def masked(mask: np.ndarray):
    return np.ma.masked_where(mask == 0, mask)


def confusion(pred_mask: np.ndarray, gt_mask: np.ndarray):
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)
    return {
        "tp": float(np.logical_and(pred, gt).sum()),
        "fp": float(np.logical_and(pred, ~gt).sum()),
        "fn": float(np.logical_and(~pred, gt).sum()),
    }


def metrics_from_confusion(values):
    tp, fp, fn = values["tp"], values["fp"], values["fn"]
    eps = 1e-7
    return {
        "dice": (2.0 * tp) / (2.0 * tp + fp + fn + eps),
        "precision": tp / (tp + fp + eps),
        "recall": tp / (tp + fn + eps),
    }


## 3. Загрузка модели и inference всех срезов


In [ ]:
checkpoint = load_checkpoint(CHECKPOINT_PATH)
config = checkpoint.get("config", {})
threshold = float(checkpoint.get("best_threshold", config.get("metrics", {}).get("threshold", 0.5)))
device = choose_device()

model = build_model(config.get("model", {})).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

images = [as_chw_float32(np.load(path, allow_pickle=False)) for path in image_paths]
gt_masks = [(as_chw_float32(np.load(path, allow_pickle=False))[0] > 0.5).astype(np.uint8) for path in mask_paths]

image_shape = images[0].shape
if any(image.shape != image_shape for image in images):
    raise ValueError("Все КТ-срезы пациента должны иметь одинаковый размер")
if any(mask.shape != image_shape[1:] for mask in gt_masks):
    raise ValueError("Размеры КТ-срезов и масок не совпадают")

probabilities = []
with torch.no_grad():
    for start in range(0, len(images), INFERENCE_BATCH_SIZE):
        batch_np = np.stack(images[start:start + INFERENCE_BATCH_SIZE])
        batch = torch.from_numpy(np.ascontiguousarray(batch_np)).to(device)
        logits = model(batch)
        probabilities.extend(torch.sigmoid(logits)[:, 0].detach().cpu().numpy().astype(np.float32))

pred_masks = [(probability >= threshold).astype(np.uint8) for probability in probabilities]
slice_metrics = [metrics_from_confusion(confusion(pred, gt)) for pred, gt in zip(pred_masks, gt_masks)]
volume_confusion = {
    key: sum(confusion(pred, gt)[key] for pred, gt in zip(pred_masks, gt_masks))
    for key in ("tp", "fp", "fn")
}
volume_metrics = metrics_from_confusion(volume_confusion)

print("device:", device)
print("checkpoint epoch:", checkpoint.get("epoch"))
print("best val dice:", checkpoint.get("best_val_dice"))
print("threshold:", threshold)
print("volume Dice:", volume_metrics["dice"])
print("volume Precision:", volume_metrics["precision"])
print("volume Recall:", volume_metrics["recall"])


## 4. Интерактивное слайд-шоу

Нажми Play или перемещай slider. Синяя область — истинная маска, красная — предсказание модели.


In [ ]:
def show_slice(index):
    image = images[index][0]
    gt_mask = gt_masks[index]
    pred_mask = pred_masks[index]
    metrics = slice_metrics[index]
    slice_name = image_paths[index].stem

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
    ax.imshow(masked(gt_mask), cmap="Blues", alpha=0.55, vmin=0, vmax=1)
    ax.imshow(masked(pred_mask), cmap="Reds", alpha=0.55, vmin=0, vmax=1)
    ax.set_title(
        f"{CASE_ID} | {slice_name} | "
        f"Dice={metrics['dice']:.3f} | "
        f"Precision={metrics['precision']:.3f} | "
        f"Recall={metrics['recall']:.3f}"
    )
    ax.legend(
        handles=[
            Patch(facecolor="tab:blue", alpha=0.55, label="Ground truth"),
            Patch(facecolor="tab:red", alpha=0.55, label="Prediction"),
        ],
        loc="lower right",
    )
    ax.axis("off")
    fig.tight_layout()
    plt.show()


play = widgets.Play(
    value=0,
    min=0,
    max=len(image_paths) - 1,
    step=1,
    interval=250,
    description="Play",
)
slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(image_paths) - 1,
    step=1,
    description="Slice",
    continuous_update=False,
)
widgets.jslink((play, "value"), (slider, "value"))
output = widgets.interactive_output(show_slice, {"index": slider})
display(widgets.HBox([play, slider]), output)
